# Cartesian versus spherical finite-cuboid FMM

This notebook changes only the expansion basis. Both plans use the full, device-resident CUDA backend with the same deterministic uniformly magnetised cuboid sources, analytically volume-averaged cuboid targets, tree, exact cuboid-to-cuboid list1 near field, FP64 precision, and exact dense direct reference. Coincident source and target cuboids retain their finite physical self-fields. Cartesian stores $N_C=(p+1)(p+2)(p+3)/6$ coefficients, while the real harmonic basis stores $N_S=(p+1)^2$. The comparison exercises the spherical cuboid P2M and volume-averaged L2P operators introduced in PR #36, alongside the established Cartesian implementation.


In [ ]:
import os
import time

import cdfmm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ORDERS = [1, 2, 3, 4, 5, 6]
GRID_AXIS = 8
TREE_DEPTH = 3
REPETITIONS = 7
CUBOID_SIDE = 0.2
MIB = 1024**2
EXECUTION_BACKEND = cdfmm.ExecutionBackend.CUDA_FULL
EXECUTION_BACKEND_NAME = "CUDA full"
M2L_EXECUTOR = "CUDA canonical target-row"
SETUP_THREADS = os.environ.get("OMP_NUM_THREADS", "runtime default")

if not cdfmm.cuda_full_available():
    raise RuntimeError(
        "The full CUDA backend is unavailable: "
        f"{cdfmm.cuda_device_description()}"
    )

CUDA_DEVICE = cdfmm.cuda_device_description()
print(
    f"Execution backend: {EXECUTION_BACKEND_NAME}; "
    f"device: {CUDA_DEVICE}; "
    f"OMP_NUM_THREADS during setup: {SETUP_THREADS}"
)


## Shared finite-cell problem

The source and target cuboids use the same regular grid, side length, and centres. The input to cdfmm is total magnetic moment, so each deterministic magnetisation is multiplied by the cuboid volume. `DenseDirectPlan` supplies the exact all-to-all cuboid-to-volume-averaged-cuboid reference. No identity map is passed because a finite cuboid self-field is physical and must be retained.


In [ ]:
axis = np.linspace(-0.875, 0.875, GRID_AXIS)
positions = np.array(
    np.meshgrid(axis, axis, axis, indexing="ij")
).reshape(3, -1).T
indices = np.arange(len(positions), dtype=np.float64)
magnetisations = np.column_stack([
    0.7 + 0.2 * np.sin(0.17 * indices),
    -0.3 + 0.25 * np.cos(0.11 * indices),
    0.4 * np.sin(0.07 * indices + 0.3),
])
cuboid = cdfmm.CuboidSize(CUBOID_SIDE, CUBOID_SIDE, CUBOID_SIDE)
cuboid_volume = CUBOID_SIDE**3
moments = np.ascontiguousarray(cuboid_volume * magnetisations)

start = time.perf_counter()
direct_plan = cdfmm.DenseDirectPlan(
    source_positions=positions,
    target_positions=positions,
    source_geometry=cdfmm.SourceGeometry.UNIFORM_CUBOID,
    target_geometry=cdfmm.TargetGeometry.VOLUME_AVERAGED_CUBOID,
    source_sizes=[cuboid],
    target_sizes=[cuboid],
    static_precision="float64",
)
direct_initialisation_s = time.perf_counter() - start

start = time.perf_counter()
H_direct = np.asarray(
    direct_plan.evaluate(
        moments,
        backend=cdfmm.DenseDirectBackend.PORTABLE,
    )
)
direct_evaluation_s = time.perf_counter() - start
updates = [
    moments * (1.0 + 0.01 * update_index)
    + 0.002 * update_index * np.roll(moments, update_index, axis=0)
    for update_index in range(REPETITIONS)
]
print(
    f"{len(positions)} source/target cuboids; "
    f"exact setup {direct_initialisation_s:.3f} s; "
    f"exact evaluation {direct_evaluation_s:.3f} s"
)


In [ ]:
def run_case(name, basis, order):
    options = cdfmm.UniformFmmOptions()
    options.expansion_basis = basis
    options.expansion_order = order
    options.precision = cdfmm.StaticPrecision.FLOAT64
    options.tree.max_level = TREE_DEPTH
    options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
    options.tree.root_half_width = 1.0
    options.source_geometry = cdfmm.SourceGeometry.UNIFORM_CUBOID
    options.source_sizes = [cuboid]
    options.target_geometry = (
        cdfmm.TargetGeometry.VOLUME_AVERAGED_CUBOID
    )
    options.target_sizes = [cuboid]
    options.use_cuboid_p2m = True
    options.backend = EXECUTION_BACKEND

    start = time.perf_counter()
    plan = cdfmm.UniformFmm(positions, positions, options)
    initialisation_s = time.perf_counter() - start

    start = time.perf_counter()
    first = plan.evaluate(moments)
    first_evaluation_s = time.perf_counter() - start

    samples = []
    stages = []
    plan.evaluate(updates[0])
    for update in updates:
        start = time.perf_counter()
        plan.evaluate(update)
        samples.append(time.perf_counter() - start)
        stages.append(dict(plan.last_timings))

    H = np.asarray(first["H"])
    difference = H - H_direct
    statistics = dict(plan.static_plan_statistics)
    cuda_statistics = dict(plan.cuda_plan_statistics)
    host_memory = statistics["total_persistent_bytes"] / MIB
    device_memory = cuda_statistics["persistent_device_bytes"] / MIB
    row = dict(
        basis=name,
        p=order,
        coefficient_count=plan.coefficient_count,
        source_geometry="uniform cuboid",
        target_geometry="volume-averaged cuboid",
        execution_backend=EXECUTION_BACKEND_NAME,
        cuda_device=CUDA_DEVICE,
        m2l_executor=M2L_EXECUTOR,
        setup_threads=SETUP_THREADS,
        rel_L2_error=np.linalg.norm(difference) / np.linalg.norm(H_direct),
        max_abs_vector_error=np.max(np.linalg.norm(difference, axis=1)),
        init_time=initialisation_s,
        first_eval_time=first_evaluation_s,
        eval_time=float(np.median(samples)),
        p2m_static_MB=statistics["p2m_operator_bytes"] / MIB,
        m2m_static_MB=statistics["m2m_operator_bytes"] / MIB,
        m2l_static_MB=statistics["m2l_operator_bytes"] / MIB,
        l2l_static_MB=statistics["l2l_operator_bytes"] / MIB,
        l2p_static_MB=statistics["l2p_operator_bytes"] / MIB,
        static_memory_MB=statistics["operator_bytes"] / MIB,
        dynamic_memory_MB=(
            statistics["multipole_state_bytes"]
            + statistics["local_state_bytes"]
        ) / MIB,
        other_state_MB=statistics["other_state_bytes"] / MIB,
        interaction_metadata_MB=statistics["interaction_bytes"] / MIB,
        scratch_MB=statistics["scratch_bytes"] / MIB,
        near_field_MB=statistics["near_field_operator_bytes"] / MIB,
        near_field_values_MB=statistics["p2p_value_bytes"] / MIB,
        near_field_metadata_MB=statistics["p2p_index_bytes"] / MIB,
        tree_metadata_MB=statistics["tree_bytes"] / MIB,
        host_memory_MB=host_memory,
        device_memory_MB=device_memory,
        cuda_setup_upload_MB=(
            cuda_statistics["setup_h2d_bytes"] / MIB
        ),
        total_memory_MB=host_memory + device_memory,
        m2l_classes=statistics["m2l_operators"],
    )
    for stage in ("p2m", "m2m", "m2l", "l2l", "l2p", "p2p"):
        row[f"{stage}_time"] = float(
            np.median([sample[stage] for sample in stages])
        )
    row["dominant_stage"] = max(
        ("p2m", "m2m", "m2l", "l2l", "l2p", "p2p"),
        key=lambda stage: row[f"{stage}_time"],
    )
    return row


rows = []
for order in ORDERS:
    rows.append(
        run_case("Cartesian", cdfmm.ExpansionBasis.CARTESIAN, order)
    )
    rows.append(
        run_case("Spherical", cdfmm.ExpansionBasis.SPHERICAL, order)
    )
    print(rows[-2])
    print(rows[-1])

results = pd.DataFrame(rows)
summary_columns = [
    "basis", "p", "coefficient_count", "source_geometry",
    "target_geometry", "execution_backend", "cuda_device",
    "m2l_executor", "setup_threads", "rel_L2_error",
    "max_abs_vector_error", "init_time", "first_eval_time",
    "eval_time", "static_memory_MB", "dynamic_memory_MB",
    "host_memory_MB", "device_memory_MB", "total_memory_MB",
    "dominant_stage",
]
memory_columns = [
    "basis", "p", "p2m_static_MB", "m2m_static_MB",
    "m2l_static_MB", "l2l_static_MB", "l2p_static_MB",
    "near_field_MB", "near_field_values_MB",
    "near_field_metadata_MB", "dynamic_memory_MB",
    "other_state_MB", "interaction_metadata_MB", "scratch_MB",
    "tree_metadata_MB", "host_memory_MB", "device_memory_MB",
    "cuda_setup_upload_MB", "total_memory_MB",
]
stage_columns = [
    "basis", "p", "p2m_time", "m2m_time", "m2l_time",
    "l2l_time", "l2p_time", "p2p_time", "dominant_stage",
]
display(results[summary_columns])
display(results[memory_columns])
display(results[stage_columns])


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
plots = [
    ("p", "rel_L2_error", "Relative L2 error vs order"),
    ("coefficient_count", "rel_L2_error", "Error vs coefficient count"),
    ("p", "eval_time", "Repeated evaluation vs order"),
    ("p", "total_memory_MB", "Persistent memory vs order"),
    ("eval_time", "rel_L2_error", "Error vs evaluation time"),
    ("total_memory_MB", "rel_L2_error", "Error vs persistent memory"),
]
for axis_plot, (x, y, title) in zip(axes.flat, plots):
    for basis, group in results.groupby("basis"):
        axis_plot.plot(group[x], group[y], "o-", label=basis)
    axis_plot.set(
        xlabel=x.replace("_", " "),
        ylabel=y.replace("_", " "),
        title=title,
    )
    if "error" in y:
        axis_plot.set_yscale("log")
    axis_plot.grid(True, alpha=0.3)
    axis_plot.legend()
fig.tight_layout()


## Interpretation

The coefficient-count plots compare degrees of freedom more fairly than order alone. Both bases use analytically volume-averaged finite-cuboid P2M and L2P operators, while exact cuboid-to-cuboid P2P and most tree metadata are common to both plans. P2M/M2M/M2L/L2L/L2P storage and multipole/local state widths change with the basis. `host_memory_MB` reports retained host plan storage, `device_memory_MB` reports the persistent CUDA allocation, and `total_memory_MB` is their sum. Inspect accuracy, `dominant_stage`, and the per-stage columns before selecting a production basis for finite-cell workloads.
